# AutoSub fine-tuning experiments

Single notebook for the Whisper fine-tuning track: load the dataset, run data-quality checks, train, evaluate, and log every run (or data fixup) to `run_log.json` so progress is tracked in one place instead of scattered scripts.

**Standalone on Colab -- no git clone needed.** Everything this notebook needs (the manifest + the audio) lives on Google Drive:

- `MyDrive/processed/` -- the Demucs output (vocals/instrumental/mix per song), already uploaded.
- `MyDrive/finetune_manifest.json` -- upload this once (it's tiny, just text). Get the latest version from the repo's `data/finetune_manifest.json` any time the dataset changes.

Everything this notebook writes (checkpoints, run_log.json, per-run train/eval logs) also goes to Drive, under `MyDrive/autosub_runs/`, so results survive when the Colab runtime disconnects/recycles -- only pull `run_log.json` back into the repo afterward to keep ARCHITECTURE.md/README.md in sync.

Run top to bottom on a GPU runtime (Runtime > Change runtime type > GPU).


## Setup


In [ ]:
# !pip install -q transformers peft datasets jiwer torchaudio wandb
# Colab's preinstalled torchao (0.10.0) is older than what peft's LoRA
# path requires (0.16.0+) -- only hit by the LoRA training path
# (get_peft_model), not the full-finetune runs, which is why this didn't
# surface until the Phase 2 LoRA ablation.
!pip install -q -U torchao

import json
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import jiwer
import requests
import torch
import torchaudio
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROCESSED_DIR = DRIVE_ROOT / "processed"  # Demucs output -- vocals/instrumental/mix per song
MANIFEST_PATH = DRIVE_ROOT / "finetune_manifest.json"  # upload from the repo's data/finetune_manifest.json

RUNS_DIR = DRIVE_ROOT / "autosub_runs"  # checkpoints + logs all land here, survives runtime resets
LOGS_DIR = RUNS_DIR / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG_PATH = LOGS_DIR / "run_log.json"
if not RUN_LOG_PATH.exists():
    RUN_LOG_PATH.write_text("[]")

SAMPLE_RATE = 16000
CHUNK_DURATION = 30.0  # seconds -- matches Whisper's native input window
CHUNK_PADDING = 0.25  # seconds -- LRCLIB line timestamps are approximate;
# padding both edges avoids clipping word onsets/codas right at the boundary.
BASELINE_MODEL_ID = "openai/whisper-base"

print("Processed audio dir:", PROCESSED_DIR, "exists:", PROCESSED_DIR.exists())
print("Manifest path:", MANIFEST_PATH, "exists:", MANIFEST_PATH.exists())
print("Logs dir:", LOGS_DIR)
print("CUDA available:", torch.cuda.is_available())

## Run log helpers

`run_log.json` is the running record of every training run _and_ every data fixup (e.g. today's duration-mismatch cleanup). Append a row after anything that changes results or the dataset -- not just training runs.


In [ ]:
def load_run_log() -> list[dict]:
    return json.loads(RUN_LOG_PATH.read_text())


def append_run_log(entry: dict) -> None:
    """Append one row to run_log.json and write it back. entry should match
    the existing schema: run, method, compute, songs, epochs, lr, train_loss,
    eval_loss, baseline_wer, finetuned_wer, notes, timestamp. Use None for
    fields that don't apply (e.g. a data-fixup row has no epochs/lr/wer).
    Adds a UTC timestamp automatically if the entry doesn't already have one.
    """
    entry = dict(entry)
    entry.setdefault("timestamp", datetime.now(timezone.utc).isoformat())
    log = load_run_log()
    log.append(entry)
    RUN_LOG_PATH.write_text(json.dumps(log, indent=2))
    print(f"Logged run '{entry.get('run')}' -- {len(log)} total entries.")


def print_run_log() -> None:
    log = load_run_log()
    header = f"{'run':<16}{'timestamp':<21}{'method':<16}{'compute':<12}{'songs':>6}{'epochs':>7}{'baseline':>10}{'finetuned':>10}{'delta':>8}"
    print(header)
    print("-" * len(header))
    for row in log:
        baseline = f"{row['baseline_wer']:.1%}" if row.get("baseline_wer") is not None else "--"
        finetuned = f"{row['finetuned_wer']:.1%}" if row.get("finetuned_wer") is not None else "--"
        if row.get("baseline_wer") is not None and row.get("finetuned_wer") is not None:
            delta = f"{row['baseline_wer'] - row['finetuned_wer']:+.1%}"
        else:
            delta = "--"
        timestamp = str(row.get("timestamp") or "--")[:19]
        print(
            f"{str(row.get('run')):<16}{timestamp:<21}{str(row.get('method')):<16}{str(row.get('compute')):<12}"
            f"{str(row.get('songs')):>6}{str(row.get('epochs')):>7}{baseline:>10}{finetuned:>10}{delta:>8}"
        )
        if row.get("notes"):
            print(f"    {row['notes']}")


print_run_log()

## Data-quality checks

`find_near_empty_segments` works standalone here (only needs the manifest). `find_duration_mismatches` needs the _raw_ pre-separation audio (to measure its real duration against LRCLIB's claim) -- that audio was never uploaded to Drive (only `processed/` vocals were), so that check only runs locally against a full repo checkout, not from this notebook. It's included here for reference/reuse if you ever run this notebook against a repo checkout instead of Drive.


In [ ]:
LRCLIB_API_URL = "https://lrclib.net/api/get"
DURATION_MISMATCH_TOLERANCE = 5.0  # seconds


def find_duration_mismatches(manifest_path: Path, repo_root: Path, tolerance: float = DURATION_MISMATCH_TOLERANCE, retries: int = 3) -> list[dict]:
    """Re-query LRCLIB for every billboard-sourced record's claimed
    recording duration, compare against the actual downloaded (raw, not
    vocals-only) audio's duration, and return records where they diverge
    by more than `tolerance` seconds -- almost certainly the wrong
    edit/upload matched to that song's lyrics (see logs/run_log.json's
    data-fixup-1 entry for how this was first found and why it matters).

    Needs repo_root pointing at a real repo checkout with data/raw/
    populated -- not runnable from this notebook's Drive-only setup, which
    never uploaded the raw pre-separation audio.
    """
    records = json.loads(manifest_path.read_text())
    mismatches = []
    for r in records:
        if r.get("source") != "billboard":
            continue
        parts = r["song_id"].split("_")
        prefix = f"billboard_{parts[1]}_{r['artist']}_"
        if not r["song_id"].startswith(prefix):
            continue
        title = r["song_id"][len(prefix):]

        for attempt in range(retries):
            try:
                resp = requests.get(
                    LRCLIB_API_URL,
                    params={"artist_name": r["artist"], "track_name": title},
                    headers={"User-Agent": "AutoSub v1.0 (dataset builder)"},
                    timeout=15,
                )
                if resp.status_code != 200:
                    time.sleep(1)
                    continue
                lrclib_duration = resp.json().get("duration", 0.0)
                audio_path = repo_root / r["audio_path"]
                waveform, sr = torchaudio.load(str(audio_path))
                audio_duration = waveform.shape[1] / sr
                if lrclib_duration and abs(audio_duration - lrclib_duration) > tolerance:
                    mismatches.append({
                        "song_id": r["song_id"],
                        "audio_duration": audio_duration,
                        "lrclib_duration": lrclib_duration,
                        "diff": abs(audio_duration - lrclib_duration),
                    })
                break
            except Exception:
                time.sleep(1)
        time.sleep(0.15)  # be polite to LRCLIB's API
    return mismatches


def find_near_empty_segments(manifest_path: Path = MANIFEST_PATH, min_chars: int = 2) -> list[dict]:
    """Find lyrics_segments with near-empty/symbol-only text (e.g. a '♪'
    placeholder for an instrumental break) -- these can span tens of
    seconds of real audio while training the model to map it to almost
    nothing. _group_into_chunks (below) already filters these out before
    chunking; this is just for visibility/auditing.
    """
    records = json.loads(manifest_path.read_text())
    suspect = []
    for r in records:
        for seg in r["lyrics_segments"]:
            text = seg["text"].strip()
            if len(text) <= min_chars:
                suspect.append({
                    "song_id": r["song_id"],
                    "start": seg["start"],
                    "end": seg["end"],
                    "duration": seg["end"] - seg["start"],
                    "text": text,
                })
    return suspect


# find_duration_mismatches needs a repo checkout, not runnable here -- see docstring.

near_empty = find_near_empty_segments()
print(f"{len(near_empty)} near-empty segments found")

## Dataset loading + chunking


In [ ]:
def _resolve_vocals_path(record: dict) -> dict:
    """The manifest stores vocals_path as a repo-relative path, e.g.
    'data/processed/htdemucs/<id>/vocals.wav'. On Drive, the audio lives
    directly under PROCESSED_DIR (no 'data/' prefix, no repo checkout) --
    so strip the 'data/processed/' prefix and resolve against PROCESSED_DIR
    instead. audio_path (pre-separation) is intentionally left alone and
    unused here -- training/eval only ever read vocals_path, and the raw
    audio was never uploaded to Drive.
    """
    record = dict(record)
    vocals_path = record["vocals_path"]
    prefix = "data/processed/"
    if vocals_path.startswith(prefix):
        vocals_path = vocals_path[len(prefix):]
    record["vocals_path"] = str(PROCESSED_DIR / vocals_path)
    return record


def load_manifest(manifest_path: Path = MANIFEST_PATH) -> tuple[list[dict], list[dict]]:
    records = [_resolve_vocals_path(r) for r in json.loads(manifest_path.read_text())]
    train_records = [r for r in records if r["split"] == "train"]
    eval_records = [r for r in records if r["split"] == "eval"]
    return train_records, eval_records


def _group_into_chunks(segments: list[dict], chunk_duration: float = CHUNK_DURATION) -> list[dict]:
    """Group a song's line-level lyrics_segments into ~chunk_duration-long
    windows, each with a combined start/end and concatenated text.
    Drops near-empty segments (text length <= 2, e.g. '♪' placeholders
    for instrumental breaks) before grouping -- some span 20+ seconds of
    real audio, which would otherwise train the model to map long stretches
    of vocals/instrumental onto a near-empty label.
    """
    segments = [s for s in segments if len(s["text"].strip()) > 2]
    if not segments:
        return []

    chunks = []
    current = [segments[0]]
    chunk_start = segments[0]["start"]

    for segment in segments[1:]:
        if segment["end"] - chunk_start > chunk_duration:
            chunks.append(current)
            current = [segment]
            chunk_start = segment["start"]
        else:
            current.append(segment)
    chunks.append(current)

    return [
        {
            "start": chunk[0]["start"],
            "end": chunk[-1]["end"],
            "text": " ".join(s["text"] for s in chunk),
        }
        for chunk in chunks
    ]


def _build_hf_dataset(records: list[dict], processor: WhisperProcessor) -> Dataset:
    """Turn manifest records into a HF Dataset with input_features + labels,
    one example per ~30s chunk (see _group_into_chunks)."""
    chunk_audio_paths, chunk_starts, chunk_ends, chunk_texts = [], [], [], []

    for record in records:
        for chunk in _group_into_chunks(record["lyrics_segments"]):
            chunk_audio_paths.append(record["vocals_path"])
            chunk_starts.append(chunk["start"])
            chunk_ends.append(chunk["end"])
            chunk_texts.append(chunk["text"])

    dataset = Dataset.from_dict({
        "audio_path": chunk_audio_paths,
        "start": chunk_starts,
        "end": chunk_ends,
        "text": chunk_texts,
    })

    def _prepare(example):
        waveform, sample_rate = torchaudio.load(example["audio_path"])
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sample_rate != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)

        start_sample = max(int((example["start"] - CHUNK_PADDING) * SAMPLE_RATE), 0)
        end_sample = min(int((example["end"] + CHUNK_PADDING) * SAMPLE_RATE), waveform.shape[1])
        chunk_waveform = waveform[0, start_sample:end_sample].numpy()

        example["input_features"] = processor.feature_extractor(
            chunk_waveform, sampling_rate=SAMPLE_RATE
        ).input_features[0]
        example["labels"] = processor.tokenizer(example["text"]).input_ids
        return example

    return dataset.map(_prepare, remove_columns=dataset.column_names)

## Training


In [ ]:
@dataclass
class _DataCollatorSpeechSeq2SeqWithPadding:
    """Pads input_features and labels independently, and masks padded
    label positions with -100 so they're ignored by the loss."""
    processor: WhisperProcessor

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels
        return batch


def make_run_key(run_name: str) -> str:
    """run_name alone isn't unique across reruns (e.g. retrying
    'whisper-full-v5' after a crash would silently overwrite the first
    attempt's logs). Appends a UTC timestamp so every run gets its own
    log files while keeping the meaningful name readable in front:
    'whisper-full-v5_20261030-0051'.
    """
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M")
    return f"{run_name}_{timestamp}"


class _TrainingLogCallback(TrainerCallback):
    """Prints step count, % done, elapsed/remaining time, current loss on
    every log event, AND appends every log event (train step + per-epoch
    eval) to LOGS_DIR/<run_key>_train.jsonl as it happens -- so progress
    survives a Colab disconnect/crash instead of only living in captured
    stdout, and every run's full epoch-by-epoch history ends up in the same
    logs folder as everything else.
    """

    def __init__(self, run_key: str):
        self.log_path = LOGS_DIR / f"{run_key}_train.jsonl"
        self.log_path.write_text("")  # start fresh for this run

    def on_train_begin(self, args, state, control, **kwargs):
        self._start_time = time.monotonic()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        elapsed = time.monotonic() - self._start_time
        record = {
            "step": state.global_step,
            "epoch": state.epoch,
            "elapsed_seconds": elapsed,
            **logs,
        }
        with self.log_path.open("a") as f:
            f.write(json.dumps(record) + "\n")

        if "loss" in logs:
            total_steps = state.max_steps
            current_step = state.global_step
            fraction_done = current_step / total_steps if total_steps else 0.0
            remaining = (elapsed / current_step) * (total_steps - current_step) if current_step else 0.0
            print(
                f"[train] step {current_step}/{total_steps} ({fraction_done:.0%}) "
                f"| loss {logs['loss']:.4f} | elapsed {_format_duration(elapsed)} "
                f"| est. remaining {_format_duration(remaining)}"
            )
        elif "eval_loss" in logs:
            print(f"[eval]  epoch {state.epoch:.0f} | eval_loss {logs['eval_loss']:.4f}")


def _format_duration(seconds: float) -> str:
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    if minutes:
        return f"{minutes}m{secs:02d}s"
    return f"{secs}s"


def _build_lora_model(base_model_id: str) -> PeftModel:
    model = WhisperForConditionalGeneration.from_pretrained(base_model_id)
    model.generation_config.forced_decoder_ids = None
    lora_config = LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj"], lora_dropout=0.05)
    return get_peft_model(model, lora_config)


def _build_full_model(base_model_id: str) -> WhisperForConditionalGeneration:
    model = WhisperForConditionalGeneration.from_pretrained(base_model_id)
    model.generation_config.forced_decoder_ids = None
    return model


def train(
    manifest_path: Path = MANIFEST_PATH,
    output_dir: Path | None = None,
    base_model_id: str = "openai/whisper-base",
    num_epochs: int = 5,
    batch_size: int = 8,
    learning_rate: float = 1e-4,
    max_steps: int = -1,
    resume_from_checkpoint: str | None = None,
    run_name: str | None = None,
    report_to_wandb: bool = False,
    full_finetune: bool = True,
) -> Path:
    """Fine-tune base_model_id on the manifest's train split, using the
    eval split for periodic validation, and save to output_dir (defaults
    to RUNS_DIR/checkpoints/<run_key> on Drive, so the checkpoint
    survives a runtime reset).

    full_finetune=True trains every weight (needs real GPU memory); False
    uses a LoRA adapter (~1.6% of params trainable, practical on MPS/laptop).

    Every train-step and per-epoch-eval log line is written live to
    LOGS_DIR/<run_key>_train.jsonl (see _TrainingLogCallback), where
    run_key is run_name + a UTC timestamp (see make_run_key) -- so rerunning
    the same run_name never overwrites a previous attempt's logs. Returns
    run_key alongside the usual result fields so the eval + logging cells
    can reuse the exact same key.
    """
    if run_name is None:
        raise ValueError("run_name is required -- it names this run's log file in LOGS_DIR.")
    run_key = make_run_key(run_name)
    if output_dir is None:
        output_dir = RUNS_DIR / "checkpoints" / run_key

    train_records, eval_records = load_manifest(manifest_path)

    processor = WhisperProcessor.from_pretrained(base_model_id, task="transcribe")
    train_dataset = _build_hf_dataset(train_records, processor)
    eval_dataset = _build_hf_dataset(eval_records, processor)

    model = _build_full_model(base_model_id) if full_finetune else _build_lora_model(base_model_id)
    data_collator = _DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        num_train_epochs=num_epochs,
        max_steps=max_steps,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch" if max_steps < 0 else "no",
        save_strategy="epoch" if max_steps < 0 else "no",
        predict_with_generate=True,
        generation_max_length=225,
        label_names=["labels"],
        remove_unused_columns=False,
        logging_steps=1,
        report_to=["wandb"] if report_to_wandb else [],
        run_name=run_key,
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        callbacks=[_TrainingLogCallback(run_key)],
        processing_class=processor,
    )

    trainer.train(resume_from_checkpoint=resume_from_checkpoint)

    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    processor.save_pretrained(output_dir)

    # last logged train/eval loss, for the run log
    train_losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    eval_losses = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]

    return {
        "run_key": run_key,
        "output_dir": output_dir,
        "final_train_loss": train_losses[-1] if train_losses else None,
        "final_eval_loss": eval_losses[-1] if eval_losses else None,
        "num_train_songs": len(train_records),
    }

In [ ]:
# Phase 2 ablation: isolate what actually drove v4/v5's win. v4 changed
# TWO variables at once vs v3 (LoRA -> full fine-tune, AND 3 -> 5 epochs),
# so it's unclear which mattered. This run keeps 5 epochs but switches
# back to LoRA (full_finetune=False) -- if this also beats baseline by a
# lot, epochs/data were doing the work and full fine-tuning wasn't
# necessary; if it's much worse than v5's 55.1%, full fine-tuning was the
# real driver.
RUN_NAME = "whisper-lora-ablation"

result = train(
    manifest_path=MANIFEST_PATH,
    num_epochs=5,
    full_finetune=False,
    run_name=RUN_NAME,
    report_to_wandb=True,
)
print(result)
print("run_key:", result["run_key"])  # e.g. whisper-lora-ablation_20261030-0312 -- used by every log file below

## Evaluation


In [ ]:
_WER_NORMALIZATION = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
])


def load_eval_records(manifest_path: Path = MANIFEST_PATH) -> list[dict]:
    records = [_resolve_vocals_path(r) for r in json.loads(manifest_path.read_text())]
    return [r for r in records if r["split"] == "eval"]


def _build_eval_chunks(records: list[dict]) -> list[dict]:
    chunks = []
    for record in records:
        for chunk in _group_into_chunks(record["lyrics_segments"], chunk_duration=CHUNK_DURATION):
            chunks.append({
                "audio_path": record["vocals_path"],
                "start": chunk["start"],
                "end": chunk["end"],
                "reference_text": chunk["text"],
            })
    return chunks


def _load_chunk_waveform(audio_path: str, start: float, end: float) -> "torch.Tensor":
    waveform, sample_rate = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sample_rate != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)
    start_sample = max(int((start - CHUNK_PADDING) * SAMPLE_RATE), 0)
    end_sample = min(int((end + CHUNK_PADDING) * SAMPLE_RATE), waveform.shape[1])
    return waveform[0, start_sample:end_sample]


def _transcribe_chunk(waveform: "torch.Tensor", model, processor: WhisperProcessor) -> str:
    inputs = processor.feature_extractor(waveform.numpy(), sampling_rate=SAMPLE_RATE, return_tensors="pt")
    with torch.inference_mode():
        predicted_ids = model.generate(inputs.input_features, max_length=225)
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()


def _load_baseline_model(model_id: str = BASELINE_MODEL_ID):
    processor = WhisperProcessor.from_pretrained(model_id, task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(model_id)
    model.generation_config.forced_decoder_ids = None
    model.eval()
    return model, processor


def _load_finetuned_model(checkpoint_dir: Path, base_model_id: str = BASELINE_MODEL_ID):
    """Auto-detects LoRA adapter checkpoints (adapter_config.json present)
    vs. full-finetune checkpoints (complete model, loaded directly)."""
    processor = WhisperProcessor.from_pretrained(checkpoint_dir, task="transcribe")
    if (Path(checkpoint_dir) / "adapter_config.json").exists():
        base_model = WhisperForConditionalGeneration.from_pretrained(base_model_id)
        base_model.generation_config.forced_decoder_ids = None
        model = PeftModel.from_pretrained(base_model, checkpoint_dir)
    else:
        model = WhisperForConditionalGeneration.from_pretrained(checkpoint_dir)
        model.generation_config.forced_decoder_ids = None
    model.eval()
    return model, processor


def evaluate_model(chunks: list[dict], model, processor: WhisperProcessor) -> dict:
    references, predictions = [], []
    for i, chunk in enumerate(chunks):
        waveform = _load_chunk_waveform(chunk["audio_path"], chunk["start"], chunk["end"])
        predictions.append(_transcribe_chunk(waveform, model, processor))
        references.append(chunk["reference_text"])
        if (i + 1) % 10 == 0:
            print(f"  evaluated {i + 1}/{len(chunks)} chunks")

    normalized_references = [_WER_NORMALIZATION(r) for r in references]
    normalized_predictions = [_WER_NORMALIZATION(p) for p in predictions]
    wer = jiwer.wer(normalized_references, normalized_predictions)
    return {"wer": wer, "predictions": predictions, "references": references}


def compare_baseline_vs_finetuned(
    manifest_path: Path,
    checkpoint_dir: Path,
    base_model_id: str = BASELINE_MODEL_ID,
    run_name: str | None = None,
) -> dict:
    """Evaluate baseline vs. fine-tuned checkpoint on the manifest's eval
    split. If run_name is given, writes the full per-chunk result (WER +
    every prediction/reference pair) to LOGS_DIR/<run_name>_eval.json --
    same logs folder as the training JSONL, so a run's train + eval history
    live side by side.
    """
    eval_records = load_eval_records(manifest_path)
    if not eval_records:
        raise ValueError(f"No eval-split records found in {manifest_path}")
    chunks = _build_eval_chunks(eval_records)

    print(f"Evaluating baseline on {len(chunks)} chunks...")
    baseline_model, baseline_processor = _load_baseline_model(base_model_id)
    baseline_result = evaluate_model(chunks, baseline_model, baseline_processor)

    print(f"Evaluating fine-tuned checkpoint on {len(chunks)} chunks...")
    finetuned_model, finetuned_processor = _load_finetuned_model(checkpoint_dir, base_model_id)
    finetuned_result = evaluate_model(chunks, finetuned_model, finetuned_processor)

    print(f"Baseline WER:    {baseline_result['wer']:.1%}")
    print(f"Fine-tuned WER:  {finetuned_result['wer']:.1%}")
    improvement = baseline_result["wer"] - finetuned_result["wer"]
    print(f"Improvement:     {improvement:+.1%} (positive = fine-tuning helped)")

    result = {"baseline": baseline_result, "finetuned": finetuned_result}

    if run_name:
        eval_log_path = LOGS_DIR / f"{run_name}_eval.json"
        eval_log_path.write_text(json.dumps(result, indent=2))
        print(f"Eval log written to {eval_log_path}")

    return result

In [ ]:
eval_result = compare_baseline_vs_finetuned(
    manifest_path=MANIFEST_PATH,
    checkpoint_dir=result["output_dir"],
    run_name=result["run_key"],
)

## Error breakdown (per-chunk diagnosis)

Substitution/deletion/insertion breakdown + repetition-loop detection per chunk, sorted by regression -- the ASR analog of a confusion matrix. Use this before guessing at the next fix.


In [ ]:
import re

_REPETITION_RE = re.compile(r"\b(\w+(?:\s+\w+){0,3})\b(?:\s+\1\b){2,}", re.IGNORECASE)


def _error_counts(reference: str, prediction: str) -> dict:
    ref_norm = _WER_NORMALIZATION(reference)
    pred_norm = _WER_NORMALIZATION(prediction)
    if not ref_norm.strip():
        return {"wer": 0.0, "substitutions": 0, "deletions": 0, "insertions": 0, "hits": 0}
    result = jiwer.process_words([ref_norm], [pred_norm])
    return {
        "wer": result.wer,
        "substitutions": result.substitutions,
        "deletions": result.deletions,
        "insertions": result.insertions,
        "hits": result.hits,
    }


def _has_repetition_loop(text: str) -> bool:
    return _REPETITION_RE.search(_WER_NORMALIZATION(text)) is not None


def analyze_errors(eval_result: dict, top_n: int = 15) -> list[dict]:
    baseline, finetuned = eval_result["baseline"], eval_result["finetuned"]
    rows = []
    for i, reference in enumerate(baseline["references"]):
        baseline_pred = baseline["predictions"][i]
        finetuned_pred = finetuned["predictions"][i]
        baseline_errors = _error_counts(reference, baseline_pred)
        finetuned_errors = _error_counts(reference, finetuned_pred)
        rows.append({
            "index": i,
            "reference": reference,
            "baseline_prediction": baseline_pred,
            "finetuned_prediction": finetuned_pred,
            "baseline_wer": baseline_errors["wer"],
            "finetuned_wer": finetuned_errors["wer"],
            "wer_delta": finetuned_errors["wer"] - baseline_errors["wer"],
            "finetuned_substitutions": finetuned_errors["substitutions"],
            "finetuned_deletions": finetuned_errors["deletions"],
            "finetuned_insertions": finetuned_errors["insertions"],
            "finetuned_repetition_loop": _has_repetition_loop(finetuned_pred),
            "baseline_repetition_loop": _has_repetition_loop(baseline_pred),
        })

    total_sub = sum(r["finetuned_substitutions"] for r in rows)
    total_del = sum(r["finetuned_deletions"] for r in rows)
    total_ins = sum(r["finetuned_insertions"] for r in rows)
    total_errors = total_sub + total_del + total_ins
    repetition_chunks = [r for r in rows if r["finetuned_repetition_loop"] and not r["baseline_repetition_loop"]]
    worst = sorted(rows, key=lambda r: r["wer_delta"], reverse=True)[:top_n]

    print(f"Error breakdown ({total_errors} total word errors on fine-tuned predictions):")
    if total_errors:
        print(f"  substitutions: {total_sub} ({total_sub / total_errors:.0%})")
        print(f"  deletions:     {total_del} ({total_del / total_errors:.0%})")
        print(f"  insertions:    {total_ins} ({total_ins / total_errors:.0%})")
    print(f"Repetition-loop artifacts introduced by fine-tuning: {len(repetition_chunks)}/{len(rows)} chunks")

    print(f"\nTop {top_n} chunks where fine-tuning made things WORSE:")
    for r in worst:
        if r["wer_delta"] <= 0:
            break
        print(f"\n[{r['index']}] baseline {r['baseline_wer']:.0%} -> finetuned {r['finetuned_wer']:.0%} (\u0394{r['wer_delta']:+.0%})")
        print(f"  ref:       {r['reference']}")
        print(f"  baseline:  {r['baseline_prediction']}")
        print(f"  finetuned: {r['finetuned_prediction']}")

    return rows


error_rows = analyze_errors(eval_result)

## Log this run

Fill in method/compute/songs/epochs/lr to match what you actually ran, then append to `run_log.json`. Commit the updated `run_log.json` alongside any ARCHITECTURE.md/README.md updates.


In [ ]:
append_run_log({
    "run": result["run_key"],
    "method": "LoRA",
    "compute": "GPU (fp16)",
    "songs": result["num_train_songs"],
    "epochs": 5,
    "lr": 1e-4,
    "train_loss": result["final_train_loss"],
    "eval_loss": result["final_eval_loss"],
    "baseline_wer": eval_result["baseline"]["wer"],
    "finetuned_wer": eval_result["finetuned"]["wer"],
    "notes": (
        "Phase 2 ablation: same clean data + 5 epochs as v5, but LoRA instead "
        "of full fine-tune -- isolates whether v4/v5's win came from full "
        "fine-tuning or just from epochs/clean data. Compare this WER against "
        "v5's 55.1% to tell which."
    ),
})

print_run_log()

## Syncing results back to the repo

This notebook's `run_log.json` and per-run logs live on Drive (`MyDrive/autosub_runs/logs/`), separate from the repo's `notebooks/logs/run_log.json`. After a run you care about, download the updated `run_log.json` from Drive and either replace or merge it into the repo's copy, then commit -- that keeps the historical record (v1-v4, data-fixup-1, etc.) and this session's new runs in one place for ARCHITECTURE.md/README.md to cite.
